# Extra · Regional analysis at different zoom levels

Often you want the mean over a **region**: a country, a mountain range, a river basin. On the HEALPix grid
(classes 05 and 06) that means: find the **cells inside the region**, and average them. In this notebook you

1. select the cells of a **box** of latitudes and longitudes, at zoom 6 and 8,
2. compare the **regional mean** at both zooms,
3. use a **polygon** for a region that is not a box, and
4. see how much the shape of the region and the zoom change the result.

Example: the **Tibetan Plateau** and its surface pressure (ERA5), which shows the height of the land very well:
the higher the ground, the lower the pressure.

## 1. Data and the position of the cells

ERA5 from the catalog of class 05, at zoom 6 (~100 km) and zoom 8 (~25 km). ERA5 has the latitude and
longitude of every cell (`lat`, `lon`). Its longitudes go from 0 to 360; the regions are easier to write
from −180 to 180.

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import easygems.healpix as egh
import intake
import matplotlib.pyplot as plt
import numpy as np

cat = intake.open_catalog("https://data.nextgems-h2020.eu/online.yaml")
era5 = {zoom: cat.ERA5(zoom=zoom).to_dask() for zoom in (6, 8)}


def positions(ds):
    # longitude (from -180 to 180) and latitude of every cell
    return (ds["lon"].values + 180) % 360 - 180, ds["lat"].values

## 2. A box

The Tibetan Plateau, roughly: 73°E to 105°E and 26°N to 40°N. A cell is inside if its position is inside all
four limits; `np.where` gives the numbers of those cells:

In [ ]:
box = {}
for zoom, ds in era5.items():
    lon, lat = positions(ds)
    inside = (lon > 73) & (lon < 105) & (lat > 26) & (lat < 40)
    box[zoom] = np.where(inside)[0]
    print(f"zoom {zoom}: {len(box[zoom]):,} cells in the box")

16 times more cells at zoom 8: every zoom level has 4 times more.

Only the cells of the region are read: the mean surface pressure of 1991–2020, in hPa. To draw them, they
are put into a map of the whole globe that is empty (`NaN`) everywhere else.

In [ ]:
def region_map(zoom, cells, variable="sp", factor=0.01):
    # mean of 1991-2020 in the cells of a region, in a map of the globe that is empty elsewhere
    ds = era5[zoom]
    values = (
        ds[variable]
        .isel(cell=cells)
        .sel(time=slice("1991", "2020"))
        .mean("time")
        .compute()
    )
    full = np.full(ds.sizes["cell"], np.nan)
    full[cells] = values * factor  # Pa to hPa
    return full


fig, axes = plt.subplots(
    1, 2, figsize=(14, 5), subplot_kw={"projection": ccrs.PlateCarree()}
)
for ax, zoom in zip(axes, (6, 8)):
    ax.set_extent([60, 120, 15, 55], crs=ccrs.PlateCarree())
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    image = egh.healpix_show(
        region_map(zoom, box[zoom]), ax=ax, cmap="viridis", vmin=500, vmax=1000
    )
    ax.set_title(f"Box, zoom {zoom}: {len(box[zoom]):,} cells")
fig.colorbar(image, ax=axes, shrink=0.7, label="surface pressure / hPa")

fig.savefig("/app/pages/regional_analysis/box_maps.png", dpi=100, bbox_inches="tight")
plt.show()

The plateau is the dark part (low pressure, high ground). The box also contains a lot of lowland around it.

## 3. The regional mean at zoom 6 and 8

All cells have the same area, so the regional mean is a plain mean over the cells, as the global mean in
class 05. Monthly, 1991–2020:

In [ ]:
def regional_mean(zoom, cells, variable="sp", factor=0.01):
    ds = era5[zoom]
    return (
        ds[variable]
        .isel(cell=cells)
        .sel(time=slice("1991", "2020"))
        .mean("cell")
        .compute()
        * factor
    )


box_mean = {zoom: regional_mean(zoom, box[zoom]) for zoom in (6, 8)}

fig, ax = plt.subplots(figsize=(10, 4))
for zoom in (6, 8):
    box_mean[zoom].plot(ax=ax, label=f"zoom {zoom}")
ax.set_ylabel("surface pressure / hPa")
ax.set_xlabel("year")
ax.set_title("Mean surface pressure of the box of the Tibetan Plateau")
ax.legend()

fig.savefig(
    "/app/pages/regional_analysis/box_timeseries.png", dpi=100, bbox_inches="tight"
)
plt.show()

difference = abs(box_mean[6] - box_mean[8]).mean()
print(f"mean absolute difference zoom 6 - zoom 8: {float(difference):.1f} hPa")

## 4. A polygon

A box is a poor shape for the plateau. A **polygon** follows its edge: a list of (longitude, latitude)
corners. `shapely` makes a shape of it, and `geopandas` tells which cells are **within** it.

In [ ]:
import geopandas as gpd
from shapely.geometry import Polygon

# the edge of the Tibetan Plateau: (longitude, latitude) of its corners
tibet = Polygon(
    [
        (69.30, 35.05),
        (71.45, 34.12),
        (73.39, 34.35),
        (77.16, 30.70),
        (82.68, 28.03),
        (87.67, 26.79),
        (92.91, 26.95),
        (95.50, 28.12),
        (96.24, 27.53),
        (98.06, 27.44),
        (98.28, 24.86),
        (100.00, 25.00),
        (103.02, 25.03),
        (104.58, 26.84),
        (103.53, 28.52),
        (103.04, 29.97),
        (104.00, 31.15),
        (104.63, 34.27),
        (103.04, 35.55),
        (102.99, 37.47),
        (101.52, 38.55),
        (99.55, 38.97),
        (96.55, 39.90),
        (92.63, 40.38),
        (89.20, 39.05),
        (84.06, 36.96),
        (80.63, 36.41),
        (76.80, 37.10),
        (74.11, 39.52),
        (70.79, 38.86),
        (70.85, 36.96),
        (69.30, 35.05),
    ]
)

polygon = {}
for zoom, ds in era5.items():
    lon, lat = positions(ds)
    points = gpd.GeoSeries(gpd.points_from_xy(lon, lat))  # one point per cell
    polygon[zoom] = np.where(points.within(tibet))[0]
    print(f"zoom {zoom}: {len(polygon[zoom]):,} cells in the polygon")

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(14, 5), subplot_kw={"projection": ccrs.PlateCarree()}
)
for ax, zoom in zip(axes, (6, 8)):
    ax.set_extent([60, 120, 15, 55], crs=ccrs.PlateCarree())
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.3)
    image = egh.healpix_show(
        region_map(zoom, polygon[zoom]), ax=ax, cmap="viridis", vmin=500, vmax=1000
    )
    ax.plot(*tibet.exterior.xy, color="red", transform=ccrs.PlateCarree())
    ax.set_title(f"Polygon, zoom {zoom}: {len(polygon[zoom]):,} cells")
fig.colorbar(image, ax=axes, shrink=0.7, label="surface pressure / hPa")

fig.savefig(
    "/app/pages/regional_analysis/polygon_maps.png", dpi=100, bbox_inches="tight"
)
plt.show()

## 5. Shape or zoom: what matters more?

The regional mean of the polygon, at both zooms, next to the box at zoom 8:

In [ ]:
polygon_mean = {zoom: regional_mean(zoom, polygon[zoom]) for zoom in (6, 8)}

fig, ax = plt.subplots(figsize=(10, 4))
box_mean[8].plot(ax=ax, label="box, zoom 8")
polygon_mean[6].plot(ax=ax, label="polygon, zoom 6")
polygon_mean[8].plot(ax=ax, label="polygon, zoom 8")
ax.set_ylabel("surface pressure / hPa")
ax.set_xlabel("year")
ax.set_title("Mean surface pressure of the Tibetan Plateau")
ax.legend()

fig.savefig(
    "/app/pages/regional_analysis/polygon_timeseries.png", dpi=100, bbox_inches="tight"
)
plt.show()

print(
    f"box - polygon (zoom 8):     {float((box_mean[8] - polygon_mean[8]).mean()):+.1f} hPa"
)
print(
    f"polygon zoom 6 - zoom 8:    {float((polygon_mean[6] - polygon_mean[8]).mean()):+.1f} hPa"
)

## 6. Exercise

1. A small region: **Costa Rica**, roughly 86°W to 82.5°W and 8°N to 11.5°N. How many cells does its box have
   at zoom 6, and at zoom 8? What does that mean for a regional mean at zoom 6?
2. Choose a region you are interested in (e.g. your region of class 01): its cells at zoom 6 and 8, a map of
   both, and the regional mean of the **2 m temperature** (`variable="2t"`, and `factor=1` with `- 273.15` for °C).
   Save your map as `/app/pages/regional_analysis/region_exercise.png`.

In [ ]:
# 1. Costa Rica

In [ ]:
# 2. your region

## Your page

Open `pages/regional_analysis/my_page.rst` in JupyterLab and describe what you found. Look at it on
http://localhost:8000 → *Extras*.